In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle
import html
import unicodedata

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /Users/win/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/win/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
def create_stem_cache(corpus):
    unique_words = set()
    for text in corpus:
        text = str(text).lower()
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
        text = re.sub(r'[^a-z\s]', ' ', text)
        unique_words.update(text.split())
    
    stem_cache = {}
    ps = PorterStemmer()
    for w in unique_words:
        stem_cache[w] = ps.stem(w)
    return stem_cache

In [3]:
class CustomPreprocessor:
    def __init__(self, stop_dict, stem_cache):
        self.stop_dict = stop_dict
        self.stem_cache = stem_cache

    def __call__(self, s):
        s = str(s).lower()
        s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')
        s = re.sub(r'[^a-z\s]', ' ', s)
        tokens = s.split()
        tokens = [w for w in tokens if w not in self.stop_dict and len(w) > 2]
        tokens = [self.stem_cache.get(w, w) for w in tokens]
        return ' '.join(tokens)

In [4]:
class BM25Transformer:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def fit(self, X):
        self.N_ = X.shape[0]
        df = np.bincount(X.indices, minlength=X.shape[1])
        self.idf_ = np.log((self.N_ - df + 0.5) / (df + 0.5) + 1.0)
        self.avgdl_ = X.sum(axis=1).mean()
        return self

    def transform(self, X):
        X = X.tocoo()
        dl = X.sum(axis=1).A1
        doc_lengths = dl[X.row]
        tf = X.data
        idf = self.idf_[X.col]
        numerator = tf * (self.k1 + 1)
        denominator = tf + self.k1 * (1 - self.b + self.b * (doc_lengths / self.avgdl_))
        data = (numerator / denominator) * idf
        return sp.csr_matrix((data, (X.row, X.col)), shape=X.shape)

In [5]:
class RecipeSearchEngine:
    def __init__(self, vectorizer, bm25_matrix, df):
        self.vectorizer = vectorizer
        self.bm25_matrix = bm25_matrix
        self.df = df

    def search(self, query, top_k=5):
        query_vec = self.vectorizer.transform([query])
        scores = self.bm25_matrix.dot(query_vec.T).toarray().flatten()
        rank = np.argsort(scores)[::-1]
        
        results = self.df.iloc[rank[:top_k]].copy()
        results['Score'] = scores[rank[:top_k]]
        return results

In [6]:
df = pd.read_csv('../data/raw/recipes.csv')

df["Name"] = df["Name"].astype(str).apply(html.unescape)
df["Description"] = df["Description"].astype(str).apply(html.unescape)
df["RecipeIngredientParts"] = df["RecipeIngredientParts"].astype(str).apply(html.unescape)
df["RecipeInstructions"] = df["RecipeInstructions"].astype(str).apply(html.unescape)

df['SearchCorpus'] = df['Name'] + ' ' + df['RecipeIngredientParts'] + ' ' + df['RecipeInstructions']

In [7]:
stop_dict = set(stopwords.words('english'))
stem_cache = create_stem_cache(df['SearchCorpus'])
my_custom_preprocessor = CustomPreprocessor(stop_dict, stem_cache)

In [8]:
count_vectorizer = CountVectorizer(preprocessor=my_custom_preprocessor)
term_counts = count_vectorizer.fit_transform(df['SearchCorpus'])

In [9]:
bm25_transformer = BM25Transformer(k1=1.5, b=0.75)
bm25_transformer.fit(term_counts)
bm25_matrix = bm25_transformer.transform(term_counts)

In [10]:
searcher = RecipeSearchEngine(
    count_vectorizer, 
    bm25_matrix, 
    df[['RecipeId', 'Name', 'Images', 'RecipeIngredientParts', 'RecipeInstructions', 'AggregatedRating']]
)

with open('../resources/recipe_search_engine.pkl', 'wb') as f:
    pickle.dump(searcher, f)
    
print("Search engine successfully rebuilt with IDs and Ratings!")

Search engine successfully rebuilt with IDs and Ratings!


In [11]:
with open('../resources/recipe_search_engine.pkl', 'rb') as f:
    searcher = pickle.load(f)
results = searcher.search("eggs mix together")
display(results[['Name', 'Score']])

results = searcher.search("Rote Grutze")
display(results[['Name', 'Score']])

,Name,Score
244555,Pumpkin Pie Cake,6.932926
85860,Rita's Pumpkin Bread,6.852010
33460,Mom's Sunday Waffles,6.841070
43615,Out of This World Pumpkin Cake :),6.802243
51378,Cheesecake Lemon Bars,6.789511


,Name,Score
163582,Barossan Rote Grutze (Red Grape Sago Pudding),32.572698
86164,Rote Grütze German Mixed Berry Pudding,26.928587
136971,Berliner Rote Grütze / Mixed Berries Compote,26.894756
243,Rote Grütze,25.740088
369549,Rote Grutze (Red Fruit Jelly),21.749489
